# 5 — Water depth / penetration

All-atom CHARMM36 binary bilayer (DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA).

Water density profile along the bilayer normal and how far water penetrates past the phosphate plane. Needs water present in the `.gro`/`.tpr`.


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy scipy matplotlib
%matplotlib inline
import os, re, itertools
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config — edit these

You have **no trajectory**, so `TRAJ = None` loads the single `.gro`
frame (a **snapshot**). When you have an `.xtc`/`.trr`, set `TRAJ` to it
and every cell below averages over frames automatically.

In [ ]:
TOPOLOGY = "md.tpr"        # topology (has bonds + masses)
STRUCTURE= "md.gro"        # single-frame structure (.gro or .pdb)
TRAJ     = None            # e.g. "md_center.xtc" when you have one; None -> use STRUCTURE

LIPIDS   = ["DOPC", "DOPA"]        # the TWO lipids in THIS system
HEAD     = "P"                     # headgroup / leaflet marker
STEP     = 1                       # frame stride (only matters with a trajectory)
OUTDIR   = "results"; os.makedirs(OUTDIR, exist_ok=True)

## Load

In [ ]:
u = mda.Universe(TOPOLOGY, TRAJ) if TRAJ else mda.Universe(TOPOLOGY, STRUCTURE)
has_bonds = len(getattr(u, "bonds", [])) > 0
present   = [r for r in LIPIDS if len(u.select_atoms(f"resname {r}")) > 0]
nframes   = len(u.trajectory)
print(f"{len(u.atoms)} atoms, {nframes} frame(s), bonds={has_bonds}")
print("Lipids present:", present)
if nframes == 1:
    print("NOTE: single frame -> snapshot only (no time averaging).")

## Water penetration

In [ ]:
WATER_SEL = "resname TIP3 SOL"; WATER_O = "name OH2 or name OW or name O"; CORE=8.0
Pat=u.select_atoms(f"name {HEAD}")
wat=u.select_atoms(f"({WATER_SEL}) and ({WATER_O})") or u.select_atoms(WATER_SEL)
NB=100; zmax=60.0; edges=np.linspace(-zmax,zmax,NB+1); ctr=0.5*(edges[1:]+edges[:-1])
pw=np.zeros(NB); pp=np.zeros(NB); nf=0; depths=[]; core=[]
for ts in u.trajectory[::STEP]:
    zc=Pat.positions[:,2].mean(); zP=Pat.positions[:,2]
    upP=zP[zP>=zc].mean(); loP=zP[zP<zc].mean(); zw=wat.positions[:,2]
    pw+=np.histogram(zw-zc,bins=edges)[0]; pp+=np.histogram(zP-zc,bins=edges)[0]
    pen=np.concatenate([upP-zw[zw>=zc], zw[zw<zc]-loP])
    depths.append(pen.max() if pen.size else np.nan); core.append(int(np.sum(np.abs(zw-zc)<CORE))); nf+=1
pw/=nf; pp/=nf; depths=np.array(depths); core=np.array(core,float)
print(f"deepest water past head plane: {np.nanmean(depths):.2f} +/- {np.nanstd(depths):.2f} A")
print(f"#waters within +/-{CORE} A of centre: {core.mean():.2f} +/- {core.std():.2f}")
plt.figure(figsize=(7,4)); plt.plot(ctr,pw,label="water"); plt.plot(ctr,pp,label="phosphate")
plt.xlabel("z rel. centre (A)"); plt.ylabel("mean count/bin"); plt.legend()
plt.grid(alpha=.3); plt.title("Water penetration"); plt.show()